# Task 1: Run B — equal L1 and SSIM loss weights
Select **Runtime → Change runtime type → GPU**, then **Runtime → Run all**. Upload `task1_run_b_source.zip` when prompted and allow Drive access.

This controlled experiment changes only the original baseline's loss balance: **alpha 0.6648628294821612 → 0.5**. The loss becomes `0.5*L1 + 0.5*(1-SSIM)`. These are equal coefficients; the two terms can still have different numerical magnitudes.

The latent returns to the baseline **8,192 values** (32×16×16, 6:1 compression). Learning rate, batch size, dropout (2.92%), architecture, seed and 60-epoch budget match the baseline. Training starts from scratch, without Optuna. Run A's larger-latent weights are not used.

Results and epoch backups use `MyDrive/GenAI_Ass1/run_b_alpha05`. On reconnect, rerun the notebook to resume. Validation uses the same all-condition cases, 408 clean controls and 1,224 blur cases. It compares against both the original baseline and completed Run A; comparison with Run A changes both alpha and latent size, so only the original-baseline comparison isolates loss balance. The test set is reserved.

The final cell downloads **task1_run_b_results.zip**.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import os, zipfile, shutil, subprocess, sys, json, sqlite3
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Choose a GPU runtime, then reconnect before training.')
print('GPU:', torch.cuda.get_device_name(0))
drive.mount('/content/drive')
PROJECT = Path('/content/ass1_run_b')
BACKUP = Path('/content/drive/MyDrive/GenAI_Ass1/run_b_alpha05')
PROJECT.mkdir(exist_ok=True)
BACKUP.mkdir(parents=True, exist_ok=True)


In [ ]:
uploaded = files.upload()  # Select colab/task1_run_b_source.zip
if len(uploaded) != 1:
    raise ValueError('Upload exactly the generated source ZIP.')
archive_name = next(iter(uploaded))
with zipfile.ZipFile(archive_name) as archive:
    for entry in archive.infolist():
        destination = (PROJECT / entry.filename).resolve()
        if not destination.is_relative_to(PROJECT.resolve()):
            raise ValueError('Invalid ZIP path')
    archive.extractall(PROJECT)
os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
# Keep Colab's CUDA-enabled PyTorch installation; do not install the CPU wheel.
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)

baseline_config = json.loads(Path('experiments/run_b/baseline_config.json').read_text())
run_config = json.loads(Path('configs/task1_run_b.json').read_text())
changed = {key for key in baseline_config.keys() | run_config.keys()
           if baseline_config.get(key) != run_config.get(key)}
assert changed == {'alpha'} and run_config['alpha'] == 0.5
assert run_config['bottleneck_dim'] == 8192
print('Verified controlled configuration:', run_config)


In [ ]:
# Restore previous results after reconnecting, and define consistent backup helpers.
for folder in ['runs', 'models', 'mlruns']:
    if (BACKUP / folder).exists():
        shutil.copytree(BACKUP / folder, PROJECT / folder, dirs_exist_ok=True)
if (BACKUP / 'mlflow.db').exists():
    shutil.copy2(BACKUP / 'mlflow.db', PROJECT / 'mlflow.db')

def backup_results():
    ignore = shutil.ignore_patterns('*.sqlite3', '*.db', '*.tmp', '*-journal', '*-wal', '*-shm')
    for folder in ['runs', 'models', 'mlruns']:
        if (PROJECT / folder).exists():
            shutil.copytree(PROJECT / folder, BACKUP / folder, dirs_exist_ok=True, ignore=ignore)
    databases = list((PROJECT / 'runs').rglob('*.sqlite3')) if (PROJECT / 'runs').exists() else []
    if (PROJECT / 'mlflow.db').exists():
        databases.append(PROJECT / 'mlflow.db')
    for source in databases:
        local_copy = Path('/content') / (source.name + '.backup')
        local_copy.unlink(missing_ok=True)
        with sqlite3.connect(str(source), timeout=60) as src, sqlite3.connect(str(local_copy)) as dest:
            src.backup(dest)
        destination = BACKUP / source.relative_to(PROJECT)
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(local_copy, destination)

def run(arguments):
    process = subprocess.Popen([sys.executable, '-u', *arguments], stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, cwd=PROJECT)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue
            if isinstance(record, dict) and 'epoch' in record:
                backup_results()
        if process.wait() != 0:
            raise RuntimeError('Command failed; inspect the output above.')
    except BaseException:
        process.terminate()
        process.wait()
        backup_results()
        raise
    backup_results()


In [ ]:
# Download data to local runtime storage; preserve official split definitions.
import urllib.request, tarfile
raw = PROJECT / 'raw_data'
raw.mkdir(exist_ok=True)
for filename in ['102flowers.tgz', 'setid.mat']:
    path = raw / filename
    if not path.exists():
        temporary = raw / (filename + '.part')
        urllib.request.urlretrieve('https://www.robots.ox.ac.uk/~vgg/data/flowers/102/' + filename, temporary)
        temporary.replace(path)
if not (raw / 'jpg' / 'image_08189.jpg').exists():
    with tarfile.open(raw / '102flowers.tgz') as archive:
        archive.extractall(raw, filter='data')
run(['prepare_dataset.py'])
# Fail before training if preparation differs from the original trained baseline.
import hashlib
baseline_data = json.loads(Path('experiments/run_b/baseline_data.json').read_text())
for filename, key in [('train.json', 'train_manifest_sha256'),
                      ('validation_manifest.json', 'validation_manifest_sha256')]:
    manifest = Path('data/prepared_v2/restoration') / filename
    assert hashlib.sha256(manifest.read_bytes()).hexdigest() == baseline_data[key], filename + ' differs from baseline'
print('Training and validation manifests match the baseline.')
run(['-m', 'unittest', 'discover', '-s', 'tests', '-v'])


## Train Run B for 60 epochs
Resume only this experiment's own `last.pt` if it exists. The training code verifies configuration and data fingerprints before resuming. Both the best validation checkpoint and last completed epoch are backed up to Drive after every epoch.


In [ ]:
FINAL = Path('runs/task1/run_b/final')
TOTAL_EPOCHS = run_config['epochs']
last = FINAL / 'last.pt'
completed = 0
if last.exists():
    saved = torch.load(last, map_location='cpu', weights_only=True)
    assert saved['config'] == run_config, 'Saved Run B configuration differs; use the matching source ZIP.'
    completed = saved['epoch']
if completed < TOTAL_EPOCHS:
    arguments = ['-m', 'task1.train', 'train', '--config', 'configs/task1_run_b.json',
                 '--epochs', str(TOTAL_EPOCHS), '--device', 'cuda', '--workers', '2', '--output', str(FINAL)]
    if last.exists():
        arguments += ['--resume', str(last)]
    run(arguments)
else:
    print('Training already completed:', completed, 'epochs')


## Validate preservation and other corruption conditions
The all-condition validation uses the original fixed manifest. The blur investigation uses the same 408 validation photos, fixed clean controls, and three blur severities as the baseline. Compare the representative crops and paired metrics as well as the averages.


In [ ]:
# Inspect validation quality first; do not claim an improvement without these results.
run(['-m', 'task1.evaluate', '--checkpoint', str(FINAL / 'best.pt'), '--split', 'val',
     '--output', 'runs/task1/run_b/validation'])
from IPython.display import display, Image
for panel in sorted(Path('runs/task1/run_b/validation').glob('example_*.png'))[:12]:
    display(Image(filename=str(panel)))


In [ ]:
run(['-m', 'task1.blur_validation', '--checkpoint', str(FINAL / 'best.pt'),
     '--output', 'runs/task1/run_b/blur_validation'])
for name in ['representative_grid.png', 'gain_distributions.png', 'largest_loss_grid.png']:
    display(Image(filename=str(Path('runs/task1/run_b/blur_validation') / name)))


In [ ]:
# Compare with both completed references. Only baseline vs Run B isolates alpha.
import pandas as pd
rows = []
for reference, prefix in [('baseline', 'baseline'), ('run_a', 'run_a')]:
    for evaluation, reference_file, current_path in [
        ('all conditions', prefix + '_validation.json', 'runs/task1/run_b/validation/results.json'),
        ('fixed blur', prefix + '_blur_validation.json', 'runs/task1/run_b/blur_validation/results.json')]:
        previous = json.loads((Path('experiments/run_b') / reference_file).read_text())['metrics']
        current = json.loads(Path(current_path).read_text())['metrics']
        assert previous.keys() == current.keys(), 'Validation groups changed'
        for group, scores in current.items():
            before = previous[group]
            assert scores['n'] == before['n'], 'Validation sample counts changed'
            rows.append({'reference': reference, 'evaluation': evaluation, 'group': group, 'n': scores['n'],
                         'reference_psnr': before['psnr'], 'run_b_psnr': scores['psnr'],
                         'psnr_change': scores['psnr'] - before['psnr'],
                         'reference_ssim': before['ssim'], 'run_b_ssim': scores['ssim'],
                         'ssim_change': scores['ssim'] - before['ssim'],
                         'l1_change': scores['l1'] - before['l1']})
comparison = pd.DataFrame(rows)
comparison.to_csv('runs/task1/run_b/comparison.csv', index=False)
display(comparison.round(4))
print('Positive PSNR/SSIM change and negative L1 change mean improvement over the indicated reference.')
backup_results()


In [ ]:
# Export and verify ONNX parity, then persist the artifacts.
run(['-m', 'task1.export', '--checkpoint', str(FINAL / 'best.pt'),
     '--output', 'models/universal_run_b.onnx'])


## Download results
This archive contains model weights, ONNX export, training history, both validation reports, image panels, baseline metrics and the comparison table. Keep a copy of the downloaded ZIP; Drive also retains the resumable training artifacts.


In [ ]:
backup_results()
output = Path('/content/task1_run_b_results.zip')
with zipfile.ZipFile(output, 'w', zipfile.ZIP_DEFLATED) as archive:
    for folder in ['models', 'runs', 'mlruns', 'experiments/run_b']:
        for path in (PROJECT / folder).rglob('*'):
            if path.is_file() and not path.name.endswith(('-journal', '-wal', '-shm', '.tmp')):
                archive.write(path, path.relative_to(PROJECT))
    archive.write(PROJECT / 'configs/task1_run_b.json', 'configs/task1_run_b.json')
    if (BACKUP / 'mlflow.db').exists():
        archive.write(BACKUP / 'mlflow.db', 'mlflow.db')
files.download(str(output))
